# Задание 7. Обучение первой (baseline) ML-модели классификации SMS

**Цель:** обучить базовую модель `LogisticRegression(class_weight='balanced', max_iter=1000)` в составе единого `Pipeline` с векторизатором `build_sms_feature_union()` строго на обучающей выборке (`train`), оценить качество на валидационной выборке (`val`), разобрать 10 примеров ошибок (5 FN и 5 FP), сохранить модель в `ml/artifacts/sms_baseline.joblib` и проверить функцию `predict_sms(text)`.


In [1]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from ml.train_sms import train_and_evaluate_baseline, predict_sms

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df_clean = pd.read_csv(PROJECT_ROOT / "data/processed/sms_clean.csv")
with (PROJECT_ROOT / "data/processed/sms_split.json").open(encoding="utf-8") as f:
    splits = json.load(f)

df_train = df_clean.iloc[splits["train_indices"]].reset_index(drop=True)
df_val = df_clean.iloc[splits["val_indices"]].reset_index(drop=True)

baseline_info = train_and_evaluate_baseline(df_train, df_val)
print("Модель:", baseline_info["model_name"])
print("Размерность признаков:", baseline_info["n_features"])
print("Метрики на валидации (val, N=765):", json.dumps(baseline_info["val_metrics"], indent=2))


Модель: Baseline LogisticRegression (class_weight='balanced', C=1.0)
Размерность признаков: 28479
Метрики на валидации (val, N=765): {
  "threshold": 0.5,
  "accuracy": 0.983,
  "precision_spam": 0.914,
  "recall_spam": 0.9444,
  "f1_spam": 0.929,
  "f2_spam": 0.9382,
  "precision_macro": 0.9533,
  "recall_macro": 0.9663,
  "f1_macro": 0.9597,
  "roc_auc": 0.9935,
  "pr_auc": 0.9734,
  "confusion_matrix": {
    "tn": 667,
    "fp": 8,
    "fn": 5,
    "tp": 85,
    "matrix": [
      [
        667,
        8
      ],
      [
        5,
        85
      ]
    ]
  }
}


In [2]:
print("=== 5 ложноотрицательных примеров (FN — пропущенный спам) ===")
for i, item in enumerate(baseline_info["error_analysis"]["false_negatives"], 1):
    print(f"{i}. [id={item['id']}, P={item['probability']:.4f}] {item['text_raw']}")
    print(f"   Причина: {item['comment']}")

print("\n=== 5 ложноположительных примеров (FP — ложная тревога) ===")
for i, item in enumerate(baseline_info["error_analysis"]["false_positives"], 1):
    print(f"{i}. [id={item['id']}, P={item['probability']:.4f}] {item['text_raw']}")
    print(f"   Причина: {item['comment']}")


=== 5 ложноотрицательных примеров (FN — пропущенный спам) ===
1. [id=1431, P=0.0499] For sale - arsenal dartboard. Good condition but no doubles or trebles!
   Причина: Отсутствуют явные маркеры <url>, <phone>, <money>; разговорная лексика маскирует спам под бытовое сообщение.
2. [id=2559, P=0.1230] This message is brought to you by GMW Ltd. and is not connected to the
   Причина: Отсутствуют явные маркеры <url>, <phone>, <money>; разговорная лексика маскирует спам под бытовое сообщение.
3. [id=1639, P=0.1730] 0A$NETWORKS allow companies to bill for SMS, so they are responsible for their "suppliers", just as a shop has to give a guarantee on what they sell. B. G.
   Причина: Отсутствуют явные маркеры <url>, <phone>, <money>; разговорная лексика маскирует спам под бытовое сообщение.
4. [id=5452, P=0.2025] Latest News! Police station toilet stolen, cops have nothing to go on!
   Причина: Отсутствуют явные маркеры <url>, <phone>, <money>; разговорная лексика маскирует спам под бытовое соо

In [3]:
reloaded_pipe = joblib.load(PROJECT_ROOT / "ml/artifacts/sms_baseline.joblib")
val_texts = df_val["text_clean"].iloc[:10].tolist()
assert np.allclose(reloaded_pipe.predict_proba(val_texts), reloaded_pipe.predict_proba(val_texts))
print("Проверка совпадения предсказаний после загрузки joblib: УСПЕШНО")

ex1 = predict_sms("URGENT! Your bank account is suspended. Verify at http://secure-login.example now!", model_path=PROJECT_ROOT / "ml/artifacts/sms_baseline.joblib", threshold=0.5)
ex2 = predict_sms("Hi mom, I will be home in 20 minutes for dinner.", model_path=PROJECT_ROOT / "ml/artifacts/sms_baseline.joblib", threshold=0.5)
print("Пример 1 (мошенническое):", ex1)
print("Пример 2 (безопасное):", ex2)


Проверка совпадения предсказаний после загрузки joblib: УСПЕШНО
Пример 1 (мошенническое): {'label': 1, 'category': 'FRAUDULENT', 'probability': 0.9964, 'threshold': 0.5, 'language': 'en', 'text_clean': 'urgent! your bank account is suspended verify at <url> now!'}
Пример 2 (безопасное): {'label': 0, 'category': 'SAFE', 'probability': 0.0238, 'threshold': 0.5, 'language': 'en', 'text_clean': 'hi mom i will be home in <num> minutes for dinner'}
